# 05 Regression Modeling — Delivery Time

Compare the promise baseline and four untuned regressors using time-series cross-validation on the training split. Reuse the saved feature sets and the preprocessing definitions from `03_feature_engineering.ipynb`.


## 0. Setup and Load Data

Inputs: the regression feature table from `load_feature_table("regression")` (run `02_preprocessing.ipynb` first) and the selected feature sets saved by Part 2 of `03_feature_engineering.ipynb` in `report/tables/regression_selected_features.json`. Preprocessing is imported from `src/features.py`.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import CV_FOLDS, RANDOM_STATE, TABLES as TABLES_DIR, set_seed
from src.features import build_preprocessor, load_feature_table

set_seed()

# Seller outcome history uses training orders only (the default history_splits).
df = load_feature_table("regression")
selected_features = json.loads(
    (TABLES_DIR / "regression_selected_features.json").read_text(encoding="utf-8")
)
print("Dataset shape:", df.shape)
print("Feature configuration loaded successfully.")


Dataset shape: (96454, 72)
Feature configuration loaded successfully.


## 1. Define Prediction Target

All four models predict `log(delivery_days / promised_delivery_days)`. This measures relative delivery performance against the platform's promised delivery time, because raw `delivery_days` shows temporal drift. Convert predictions back to delivery days before calculating any evaluation metric.


In [2]:
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])

assert df["delivery_days"].notna().all()
assert df["promised_delivery_days"].notna().all()
assert (df["delivery_days"] > 0).all()
assert (df["promised_delivery_days"] > 0).all()

df["target_log_ratio"] = np.log(
    df["delivery_days"] / df["promised_delivery_days"]
)

def to_delivery_days(pred_log_ratio, promised_days):
    return promised_days * np.exp(pred_log_ratio)


## 2. Prepare Time-Ordered Train / Validation / Test

Preserve the existing split labels and chronological order. Validation and test are reserved for the later sections.


In [3]:
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

assert not train_df.empty and not val_df.empty and not test_df.empty
assert train_df["order_purchase_timestamp"].max() <= val_df["order_purchase_timestamp"].min()
assert val_df["order_purchase_timestamp"].max() <= test_df["order_purchase_timestamp"].min()

split_summary = pd.DataFrame([
    {
        "split": name,
        "n": len(frame),
        "start_date": frame["order_purchase_timestamp"].min().date(),
        "end_date": frame["order_purchase_timestamp"].max().date(),
    }
    for name, frame in [
        ("train", train_df), ("validation", val_df), ("test", test_df)
    ]
])
display(split_summary)


,split,n,start_date,end_date
0,train,83947,2016-09-15,2018-06-30
1,validation,6156,2018-07-01,2018-07-31
2,test,6351,2018-08-01,2018-08-29


## 3. Load Selected Feature Sets

Use both configurations exactly as saved in the JSON. Deduplicate only the original input column lists; keep each transformation group intact.


In [4]:
def original_columns(configuration):
    return list(dict.fromkeys(
        column for columns in configuration.values() for column in columns
    ))

linear_features = original_columns(selected_features["linear"])
tree_features = original_columns(selected_features["tree"])

print("Linear selected features:", len(linear_features))
print("Tree selected features:", len(tree_features))


Linear selected features: 33
Tree selected features: 37


## 4. Build Preprocessing Pipelines

Reuse `build_preprocessor` from 03 without changing its definitions. Preprocessors are fitted only inside model pipelines on each fold's training rows. Fixed band definitions stay unchanged; rare-category mappings are learned from fold training only.


### 4.1 Linear Preprocessing

The existing helper applies median imputation and `StandardScaler` to regular numeric columns; median imputation, `log1p` and scaling to skewed columns; and fixed bands followed by one-hot encoding to banded columns. Categorical preprocessing groups rare categories with fewer than 20 training observations and uses one-hot encoding that safely handles unknown categories.

`estimated_mean_distance_km` intentionally appears as both a skewed continuous feature and a banded feature. Both representations are retained.


In [5]:
linear_preprocessor = build_preprocessor(
    **selected_features["linear"]
)


### 4.2 Tree Preprocessing

Tree splits learn nonlinear thresholds directly, so the saved tree configuration needs no scaling, `log1p` or artificial banding. Reuse the helper with `scale=False`. Decision Tree, Random Forest and XGBoost share the same preprocessing definition, with independent copies fitted inside their pipelines.


In [6]:
tree_preprocessor = build_preprocessor(
    **selected_features["tree"],
    scale=False,
)


## 5. Define Promise Baseline

As in 03, the baseline predicts the median training log ratio. In delivery days this is `promised_delivery_days × exp(median training log ratio)`. Re-estimate the median separately within every CV fold.


In [7]:
class PromiseBaseline(RegressorMixin, BaseEstimator):
    """Predict a constant log ratio estimated from the fold's training target."""

    def fit(self, X, y):
        self.median_ = float(np.median(y))
        return self

    def predict(self, X):
        return np.full(len(X), self.median_)

promise_baseline = PromiseBaseline()


## 6. Define Untuned Candidate Models

Use default model hyperparameters, setting only the random seed, parallelism and XGBoost regression objective where relevant. Each family starts from its simplest variant: Linear Regression for the linear family, a single Decision Tree for the tree family. Random Forest and XGBoost must improve on the Decision Tree to justify their extra complexity. All models learn the same log-ratio target.


### 6.1 Linear Regression


In [8]:
linear_model = Pipeline([
    ("preprocessor", clone(linear_preprocessor)),
    ("model", LinearRegression()),
])


### 6.2 Decision Tree

In [9]:
decision_tree_model = Pipeline([
    ("preprocessor", clone(tree_preprocessor)),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])

### 6.3 Random Forest


In [10]:
random_forest_model = Pipeline([
    ("preprocessor", clone(tree_preprocessor)),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])


### 6.4 XGBoost


In [11]:
xgboost_model = Pipeline([
    ("preprocessor", clone(tree_preprocessor)),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])


## 7. Untuned Model Comparison

Evaluate only `train_df`. MAE, RMSE and bias are in delivery days; R² is also computed against actual delivery days. Bias is `mean(predicted_days - actual_days)`: positive values indicate overprediction and negative values indicate underprediction.


### 7.1 Time-Series Cross-Validation

Materialize five expanding-window folds once and reuse those exact boundaries for all five models. Each fold clones and fits its model, including preprocessing or the baseline median, using only the earlier training portion.


In [12]:
tscv = TimeSeriesSplit(n_splits=CV_FOLDS)
cv_folds = list(tscv.split(train_df))

candidate_models = {
    "Promise Baseline": (promise_baseline, ["promised_delivery_days"]),
    "Linear Regression": (linear_model, linear_features),
    "Decision Tree": (decision_tree_model, tree_features),
    "Random Forest": (random_forest_model, tree_features),
    "XGBoost": (xgboost_model, tree_features),
}

def evaluate_time_series_cv(models, frame, folds):
    """Return fold metrics in delivery days using shared chronological splits."""
    records = []
    for name, (model, columns) in models.items():
        for fold, (fit_indices, eval_indices) in enumerate(folds, start=1):
            fold_train = frame.iloc[fit_indices]
            fold_eval = frame.iloc[eval_indices]
            fitted_model = clone(model).fit(
                fold_train[columns], fold_train["target_log_ratio"]
            )
            pred_log_ratio = fitted_model.predict(fold_eval[columns])
            pred_days = to_delivery_days(
                pred_log_ratio, fold_eval["promised_delivery_days"].to_numpy()
            )
            actual_days = fold_eval["delivery_days"].to_numpy()
            records.append({
                "model": name,
                "fold": fold,
                "mae": mean_absolute_error(actual_days, pred_days),
                "rmse": np.sqrt(mean_squared_error(actual_days, pred_days)),
                "r2": r2_score(actual_days, pred_days),
                "bias": np.mean(pred_days - actual_days),
            })
    return pd.DataFrame(records)

cv_results = evaluate_time_series_cv(candidate_models, train_df, cv_folds)


### 7.2 Compare Baseline vs Untuned Models

CV mean MAE is the primary untuned comparison metric. CV standard deviation indicates temporal stability. Count a win only when the model's MAE is strictly lower than the baseline on the same fold. Random Forest and XGBoost are also compared, fold by fold, with the simplest model of their family (Decision Tree). Validation and test remain untouched at this stage.


In [13]:
cv_summary = cv_results.groupby("model").agg(
    mae_mean=("mae", "mean"),
    mae_sd=("mae", "std"),
    rmse_mean=("rmse", "mean"),
    r2_mean=("r2", "mean"),
    bias_mean=("bias", "mean"),
)

fold_mae = cv_results.pivot(index="fold", columns="model", values="mae")
fold_wins = fold_mae.lt(fold_mae["Promise Baseline"], axis=0).sum()
cv_summary["folds_beating_baseline"] = fold_wins.reindex(cv_summary.index)
cv_summary.loc["Promise Baseline", "folds_beating_baseline"] = np.nan

family_baseline = {"Random Forest": "Decision Tree", "XGBoost": "Decision Tree"}
cv_summary["folds_beating_family_baseline"] = pd.Series({
    model: fold_mae[model].lt(fold_mae[baseline]).sum()
    for model, baseline in family_baseline.items()
})

baseline_mean_mae = cv_summary.loc["Promise Baseline", "mae_mean"]
cv_summary["mae_improvement_pct"] = (
    (baseline_mean_mae - cv_summary["mae_mean"]) / baseline_mean_mae * 100
)

cv_summary = (
    cv_summary.sort_values("mae_mean")
    .rename(columns={
        "mae_mean": "CV MAE Mean",
        "mae_sd": "CV MAE SD",
        "rmse_mean": "CV RMSE Mean",
        "r2_mean": "CV R2 Mean",
        "bias_mean": "CV Bias Mean",
        "folds_beating_baseline": "Folds Beating Baseline",
        "folds_beating_family_baseline": "Folds Beating Decision Tree",
        "mae_improvement_pct": "MAE improvement vs baseline (%)",
    })
    .rename_axis("Model")
    .reset_index()
)
display(cv_summary.round(3))


,Model,CV MAE Mean,CV MAE SD,CV RMSE Mean,CV R2 Mean,CV Bias Mean,Folds Beating Baseline,Folds Beating Decision Tree,MAE improvement vs baseline (%)
0,Random Forest,5.169,0.955,8.312,0.217,-1.591,5.0,5.0,10.180
1,XGBoost,5.183,0.937,8.352,0.209,-1.632,5.0,5.0,9.927
2,Linear Regression,5.210,0.864,8.523,0.171,-2.365,4.0,NaN,9.462
3,Promise Baseline,5.755,1.065,9.167,0.049,-2.448,NaN,NaN,0.000
4,Decision Tree,7.716,0.910,13.381,-1.185,0.202,0.0,NaN,-34.076


## 8. Hyperparameter Tuning

Each model is tuned on the same five time-ordered folds as section 7 (`cv_folds`), using only `train_df`. The search scores **MAE in delivery days**: the models learn the log-ratio target, so the scorer converts predictions and targets back to days with `promised_delivery_days` before computing the error. This tunes for the number that is reported, not for the log-ratio error.

- **Lasso** replaces plain Linear Regression as the regularised linear model; `alpha` is searched on a grid.
- **Decision Tree** is tuned too, so the tree family is compared against a fair single-tree baseline rather than an overfitted full-depth tree.
- **Random Forest** and **XGBoost** use a randomised search with a fixed seed, because each fit takes 5–40 seconds.

Validation and test are not used.

In [14]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


def mae_days(estimator, X, y_log_ratio):
    """Negative MAE in days for a model fitted on the log-ratio target (higher is better)."""
    promised = X["promised_delivery_days"].to_numpy()
    pred_days = to_delivery_days(estimator.predict(X), promised)
    actual_days = to_delivery_days(np.asarray(y_log_ratio), promised)
    return -mean_absolute_error(actual_days, pred_days)


X_train_linear = train_df[linear_features]
X_train_tree = train_df[tree_features]
y_train_ratio = train_df["target_log_ratio"]

assert "promised_delivery_days" in linear_features and "promised_delivery_days" in tree_features


def search_summary(search, top=5):
    """Best candidates of a fitted search, MAE in days."""
    results = pd.DataFrame(search.cv_results_)
    columns = [c for c in results if c.startswith("param_")]
    table = results[columns + ["mean_test_score", "std_test_score"]].copy()
    table["CV MAE Mean"] = -table.pop("mean_test_score")
    table["CV MAE SD"] = table.pop("std_test_score")
    table.columns = [c.removeprefix("param_model__") for c in table.columns]
    return table.sort_values("CV MAE Mean").head(top).round(4)


best_params = {}

### 8.1 Lasso

`alpha` from $10^{-4}$ to $10$ (30 values, log-spaced), the range in the feature engineering plan. The largest values shrink every coefficient to zero, so the grid spans everything from nearly ordinary least squares to a constant prediction. Part 2 of `03_feature_engineering.ipynb` chose its `alpha` by the log-ratio error on all linear candidates; here it is re-tuned on the 33 selected features and scored in days.

In [15]:
lasso_model = Pipeline([
    ("preprocessor", clone(linear_preprocessor)),
    ("model", Lasso(max_iter=20000)),
])

lasso_search = GridSearchCV(
    lasso_model,
    {"model__alpha": np.logspace(-4, 1, 30)},
    scoring=mae_days,
    cv=cv_folds,
    n_jobs=-1,
    refit=False,
).fit(X_train_linear, y_train_ratio)

best_params["Lasso"] = lasso_search.best_params_
print("Best alpha:", lasso_search.best_params_["model__alpha"])
display(search_summary(lasso_search))

Best alpha: 0.0016102620275609393


,alpha,CV MAE Mean,CV MAE SD
7,0.0016,5.1547,0.8722
6,0.0011,5.1599,0.8538
8,0.0024,5.1661,0.8757
5,0.0007,5.1710,0.8304
9,0.0036,5.1715,0.8938


### 8.2 Decision Tree

The untuned tree grows to full depth and fits noise (CV R² below zero in section 7). `max_depth` and `min_samples_leaf` limit how far it splits; the grid covers shallow to deep trees and leaves from single orders to 200 orders.

In [16]:
decision_tree_search = GridSearchCV(
    decision_tree_model,
    {
        "model__max_depth": [4, 6, 8, 10, 12, 15, 20],
        "model__min_samples_leaf": [1, 10, 50, 100, 200],
    },
    scoring=mae_days,
    cv=cv_folds,
    n_jobs=-1,
    refit=False,
).fit(X_train_tree, y_train_ratio)

best_params["Decision Tree"] = decision_tree_search.best_params_
print("Best parameters:", decision_tree_search.best_params_)
display(search_summary(decision_tree_search))

Best parameters: {'model__max_depth': 8, 'model__min_samples_leaf': 100}


,max_depth,min_samples_leaf,CV MAE Mean,CV MAE SD
13,8,100,5.3645,0.8276
14,8,200,5.3686,0.8303
12,8,50,5.3739,0.8170
18,10,100,5.3800,0.8363
19,10,200,5.3803,0.8401


### 8.3 Random Forest

Randomised search over 15 candidates. `n_estimators` is fixed at 200: more trees only reduce variance and cost time, so it is not a parameter to tune. `min_samples_leaf` and `max_depth` control how far each tree fits the training rows; `max_features` controls how different the trees are from each other.

In [17]:
random_forest_search = RandomizedSearchCV(
    clone(random_forest_model).set_params(model__n_estimators=200),
    {
        "model__max_depth": [None, 15, 20, 30],
        "model__min_samples_leaf": [1, 5, 10, 20, 50, 100],
        "model__max_features": [0.2, 0.33, 0.5, 0.7],
    },
    n_iter=15,
    scoring=mae_days,
    cv=cv_folds,
    random_state=RANDOM_STATE,
    refit=False,
).fit(X_train_tree, y_train_ratio)

best_params["Random Forest"] = {
    "model__n_estimators": 200, **random_forest_search.best_params_
}
print("Best parameters:", best_params["Random Forest"])
display(search_summary(random_forest_search))

Best parameters: {'model__n_estimators': 200, 'model__min_samples_leaf': 5, 'model__max_features': 0.33, 'model__max_depth': 30}


,min_samples_leaf,max_features,max_depth,CV MAE Mean,CV MAE SD
5,5,0.33,30,5.1174,0.8502
2,5,0.20,30,5.1258,0.8460
8,1,0.20,None,5.1260,0.8356
0,10,0.33,30,5.1274,0.8497
14,5,0.33,15,5.1308,0.8530


### 8.4 XGBoost

Randomised search over 25 candidates. `learning_rate` and `n_estimators` trade off together (a smaller step needs more trees); `max_depth` and `min_child_weight` limit tree complexity; `subsample` and `colsample_bytree` fit each tree on a random share of rows and columns, which reduces overfitting.

In [18]:
xgboost_search = RandomizedSearchCV(
    xgboost_model,
    {
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__n_estimators": [200, 400, 600, 800],
        "model__max_depth": [3, 4, 6, 8],
        "model__min_child_weight": [1, 5, 10, 20],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.5, 0.7, 1.0],
    },
    n_iter=25,
    scoring=mae_days,
    cv=cv_folds,
    random_state=RANDOM_STATE,
    refit=False,
).fit(X_train_tree, y_train_ratio)

best_params["XGBoost"] = xgboost_search.best_params_
print("Best parameters:", xgboost_search.best_params_)
display(search_summary(xgboost_search))

Best parameters: {'model__subsample': 1.0, 'model__n_estimators': 800, 'model__min_child_weight': 20, 'model__max_depth': 8, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.7}


,subsample,n_estimators,min_child_weight,max_depth,learning_rate,colsample_bytree,CV MAE Mean,CV MAE SD
16,1.0,800,20,8,0.03,0.7,5.0469,0.8575
24,0.8,800,10,8,0.01,1.0,5.0484,0.8646
9,0.6,800,20,4,0.05,0.7,5.0521,0.8676
1,0.8,400,10,6,0.05,0.5,5.0539,0.8615
22,0.6,400,10,6,0.05,1.0,5.0595,0.8598


## 9. Tuned Model Comparison

### 9.1 Time-Series Cross-Validation

Refit each model with its best parameters on the same folds as section 7, so the fold-by-fold results are directly comparable. Linear Regression and the promise baseline have no hyperparameters and keep their section 7 results.

In [19]:
tuned_models = {
    "Lasso": (clone(lasso_model).set_params(**best_params["Lasso"]), linear_features),
    "Decision Tree (tuned)": (
        clone(decision_tree_model).set_params(**best_params["Decision Tree"]), tree_features
    ),
    "Random Forest (tuned)": (
        clone(random_forest_model).set_params(**best_params["Random Forest"]), tree_features
    ),
    "XGBoost (tuned)": (
        clone(xgboost_model).set_params(**best_params["XGBoost"]), tree_features
    ),
}

cv_results_tuned = evaluate_time_series_cv(tuned_models, train_df, cv_folds)
cv_results_all = pd.concat([cv_results, cv_results_tuned], ignore_index=True)

### 9.2 Compare Untuned vs Tuned Models

Every model is compared, fold by fold, with the promise baseline and with the simplest model of its own family: Lasso with Linear Regression, and the tuned Random Forest and XGBoost with the tuned Decision Tree. A more complex model is only worth keeping if it wins on most folds, not just on the mean, because the fold-to-fold spread (about 0.9 days) is much larger than the gaps between the best models.

In [20]:
fold_mae_all = cv_results_all.pivot(index="fold", columns="model", values="mae")

family_baseline_tuned = {
    "Lasso": "Linear Regression",
    "Random Forest (tuned)": "Decision Tree (tuned)",
    "XGBoost (tuned)": "Decision Tree (tuned)",
}

tuned_summary = cv_results_all.groupby("model").agg(
    mae_mean=("mae", "mean"),
    mae_sd=("mae", "std"),
    rmse_mean=("rmse", "mean"),
    r2_mean=("r2", "mean"),
    bias_mean=("bias", "mean"),
)
tuned_summary["folds_beating_baseline"] = (
    fold_mae_all.lt(fold_mae_all["Promise Baseline"], axis=0).sum()
)
tuned_summary.loc["Promise Baseline", "folds_beating_baseline"] = np.nan
tuned_summary["family_baseline"] = pd.Series(family_baseline_tuned)
tuned_summary["folds_beating_family_baseline"] = pd.Series({
    model: fold_mae_all[model].lt(fold_mae_all[baseline]).sum()
    for model, baseline in family_baseline_tuned.items()
})

tuned_summary = (
    tuned_summary.sort_values("mae_mean")
    .rename(columns={
        "mae_mean": "CV MAE Mean",
        "mae_sd": "CV MAE SD",
        "rmse_mean": "CV RMSE Mean",
        "r2_mean": "CV R2 Mean",
        "bias_mean": "CV Bias Mean",
        "folds_beating_baseline": "Folds Beating Baseline",
        "family_baseline": "Family Baseline",
        "folds_beating_family_baseline": "Folds Beating Family Baseline",
    })
    .rename_axis("Model")
)
display(tuned_summary.round(3))

,CV MAE Mean,CV MAE SD,CV RMSE Mean,CV R2 Mean,CV Bias Mean,Folds Beating Baseline,Family Baseline,Folds Beating Family Baseline
Model,,,,,,,,
XGBoost (tuned),5.047,0.959,8.207,0.237,-1.748,5.0,Decision Tree (tuned),5.0
Random Forest (tuned),5.117,0.951,8.289,0.222,-1.739,5.0,Decision Tree (tuned),5.0
Lasso,5.155,0.975,8.468,0.187,-2.247,5.0,Linear Regression,2.0
Random Forest,5.169,0.955,8.312,0.217,-1.591,5.0,NaN,NaN
XGBoost,5.183,0.937,8.352,0.209,-1.632,5.0,NaN,NaN
Linear Regression,5.210,0.864,8.523,0.171,-2.365,4.0,NaN,NaN
Decision Tree (tuned),5.365,0.925,8.577,0.165,-1.741,5.0,NaN,NaN
Promise Baseline,5.755,1.065,9.167,0.049,-2.448,NaN,NaN,NaN
Decision Tree,7.716,0.910,13.381,-1.185,0.202,0.0,NaN,NaN


In [21]:
# Does tuning help each model, and does the best tree model beat the tuned linear model?
untuned_vs_tuned = pd.DataFrame({
    model: {
        "Untuned CV MAE": fold_mae_all[untuned].mean(),
        "Tuned CV MAE": fold_mae_all[tuned].mean(),
        "Folds improved by tuning": int(fold_mae_all[tuned].lt(fold_mae_all[untuned]).sum()),
    }
    for model, (untuned, tuned) in {
        "Linear (Linear Regression -> Lasso)": ("Linear Regression", "Lasso"),
        "Decision Tree": ("Decision Tree", "Decision Tree (tuned)"),
        "Random Forest": ("Random Forest", "Random Forest (tuned)"),
        "XGBoost": ("XGBoost", "XGBoost (tuned)"),
    }.items()
}).T
display(untuned_vs_tuned.round(3))

best_tree = min(
    ["Random Forest (tuned)", "XGBoost (tuned)"], key=lambda m: fold_mae_all[m].mean()
)
print(
    f"{best_tree} beats Lasso in "
    f"{int(fold_mae_all[best_tree].lt(fold_mae_all['Lasso']).sum())} of {CV_FOLDS} folds; "
    f"mean MAE difference {fold_mae_all['Lasso'].mean() - fold_mae_all[best_tree].mean():.3f} days."
)
display(fold_mae_all[["Promise Baseline", "Linear Regression", "Lasso",
                      "Decision Tree (tuned)", "Random Forest (tuned)", "XGBoost (tuned)"]].round(3))

,Untuned CV MAE,Tuned CV MAE,Folds improved by tuning
Linear (Linear Regression -> Lasso),5.210,5.155,2.0
Decision Tree,7.716,5.365,5.0
Random Forest,5.169,5.117,4.0
XGBoost,5.183,5.047,5.0


XGBoost (tuned) beats Lasso in 5 of 5 folds; mean MAE difference 0.108 days.


model,Promise Baseline,Linear Regression,Lasso,Decision Tree (tuned),Random Forest (tuned),XGBoost (tuned)
fold,,,,,,
1,4.203,4.312,3.986,3.988,3.791,3.813
2,6.315,5.877,5.895,5.946,5.764,5.764
3,6.136,5.622,5.630,5.837,5.634,5.587
4,6.920,6.001,6.043,6.203,5.960,5.855
5,5.199,4.239,4.220,4.850,4.439,4.215


**Findings (time-ordered CV on train, MAE in days):**

- **Tuning helps every model**, but by different amounts: the Decision Tree drops from 7.72 to 5.37 days (5 of 5 folds), XGBoost from 5.18 to 5.05 (5 of 5), Random Forest from 5.17 to 5.12 (4 of 5).
- **Tuned XGBoost is the best model** (5.05 days, 12% below the promise baseline's 5.76). It beats the tuned Decision Tree in 5 of 5 folds, so the extra complexity is justified within the tree family.
- **Lasso does not clearly improve on Linear Regression.** Its mean is lower (5.16 vs 5.21) but it wins only 2 of 5 folds; almost all of the gain is in fold 1, the smallest training window, where regularisation matters most.
- **XGBoost beats Lasso in 5 of 5 folds, but by only 0.11 days (about 2%) on average.** The fold-to-fold spread (about 0.9 days) is much larger than the gap, so Lasso remains a reasonable, more interpretable alternative.
- **The best XGBoost parameters sit at the edge of the search space** (`n_estimators=800`, `max_depth=8` and `min_child_weight=20` are the largest values tried), so a wider search may improve it further.
- **Every model still predicts about 1.7–2.4 days too fast on average** (negative bias). Converting a log prediction back to days gives roughly the median delivery time, which is below the mean because a few deliveries take very long; MAE, the selection metric, is not affected.


## 10. Validation Evaluation


## 11. Final Model Selection


## 12. Refit on Train + Validation


## 13. Final Test Evaluation


## 14. Error Analysis


## 15. Model Interpretation and Feature Importance


## 16. Save Results
